# NexoAula - Agente educativo con RAG
Versión didáctica basada en Gemini, LangChain, FAISS y LangGraph.


## 1. Instalar dependencias


In [ ]:
!pip install -q langchain langchain-community langchain-google-genai langchain-text-splitters langgraph faiss-cpu pymupdf


## 2. Configurar la clave de Gemini en los secretos de Colab


In [ ]:
from google.colab import userdata
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')


## 3. Subir los PDF de `docs/knowledge_base` a `/content` y cargarlos


In [ ]:
from pathlib import Path
from langchain_community.document_loaders import PyMuPDFLoader
docs=[]
for p in Path('/content').glob('*.pdf'):
    docs.extend(PyMuPDFLoader(str(p)).load())
print('Páginas cargadas:', len(docs))


## 4. Dividir en fragmentos


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter=RecursiveCharacterTextSplitter(chunk_size=900, chunk_overlap=150)
chunks=splitter.split_documents(docs)
print('Chunks:', len(chunks))


## 5. Crear embeddings e índice FAISS


In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
emb=GoogleGenerativeAIEmbeddings(model='models/gemini-embedding-001', google_api_key=GEMINI_API_KEY)
store=FAISS.from_documents(chunks, emb)
retriever=store.as_retriever(search_type='similarity_score_threshold', search_kwargs={'score_threshold':0.30,'k':5})


## 6. Consultar el RAG


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage, HumanMessage
llm=ChatGoogleGenerativeAI(model='gemini-2.5-flash',temperature=0,google_api_key=GEMINI_API_KEY)
def preguntar(q):
    found=retriever.invoke(q)
    if not found: return 'No encontré esa información en los documentos disponibles.', []
    context='\n\n'.join(d.page_content for d in found)
    r=llm.invoke([SystemMessage(content='Responde solo con el contexto. Si falta información, indícalo.'),HumanMessage(content=f'Contexto: {context}\n\nPregunta: {q}')])
    return r.content, found
respuesta, fuentes=preguntar('¿Qué necesito para obtener el certificado?')
print(respuesta)


## 7. Siguiente paso
Abre el código de `src/agent.py` para ver cómo LangGraph añade triaje, solicitud de información y apertura de tickets.
